In [0]:
pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


##Running QA on Charge Reports using LLM

## QA Charge Report Extraction (LLM Mode) — Process Explanation

### Command

```bash
python3 qa_charge_report_extraction.py --mode llm --sample 10
```

### Purpose

This command runs the QA validation script in **LLM extraction mode** on a **random sample of 10 charge reports**. Instead of using regex patterns, it sends each charge report document to a Large Language Model (Claude) via the LiteLLM API to extract structured fields. This provides richer, more semantically-aware extraction compared to the default regex mode.

---

### Process Flow

```
ChargeReports/*.md  →  [Random Sample 10]  →  [LLM Extraction]  →  [Validation]  →  QA Report
    (Bronze)                                    (LiteLLM API)        (QA Gate)       (stdout)
```

### What Each Flag Does

| Flag | Purpose |
|------|---------|
| `--mode llm` | Use LLM-based extraction instead of regex. Sends each charge report to the LiteLLM endpoint for field extraction via prompt. |
| `--sample 10` | Randomly select 10 charge reports from the full set of 5,411. Useful for testing before running on all reports. |

---

### Step-by-Step Breakdown

#### 1. Configuration
The script connects to the LiteLLM endpoint:
- **Endpoint:** `https://litellm.<dns>/`
- **Model:** `agc-llmaas-claude-4-5-dev`
- Uses the OpenAI-compatible API client with the configured API key

#### 2. Sampling
From the 5,411 charge report files, 10 are randomly selected (with a fixed seed of 42 for reproducibility).

#### 3. LLM Extraction (per file)
For each selected charge report, the full markdown content is sent to the LLM with a structured prompt requesting JSON output. The LLM extracts:

| Field | Description | Regex Can Extract? |
|-------|-------------|-------------------|
| Case | SC case number | ✅ Yes |
| Charge | Charge number | ✅ Yes |
| Accused Name | Full name | ✅ Yes |
| Accused Age | Age as integer | ✅ Yes |
| Accused Gender | M or F | ✅ Yes |
| Victim Name | From Victim Particulars section only | ✅ Yes |
| Victim Age / Gender | From Victim Particulars section only | ✅ Yes |
| **Accused Relationship to Victim** | e.g., "Coach", "Neighbour", "Parent" | ❌ LLM only |
| Offence Group | Classification of offence | ⚠️ Regex uses rule mapping; LLM infers semantically |
| Special Type | e.g., "Family Violence", "LT1 Offences" | ✅ Yes |

The key advantage of LLM mode is the ability to extract the **accused's relationship to the victim** — a contextual field that requires semantic understanding of the Statement of Offence and cannot be reliably captured by regex.

#### 4. Prompt Design
The LLM prompt explicitly instructs the model to:
- Only extract victim information from the **"Victim Particulars" section** (prevents hallucinating victims from the Statement of Offence)
- Return null for missing fields rather than inferring/guessing
- Return valid JSON only

#### 5. Validation
The same validation rules apply as in regex mode:
- Required fields must be present
- Format checks on case/charge numbers
- Age range validation (7–120)
- Gender value validation (M/F)
- Victim field consistency

#### 6. Output
The QA report is printed to stdout showing:
- Number of reports processed
- Extraction failures (if any API calls failed)
- All validation issues with severity (ERROR/WARNING)

---

### When to Use LLM Mode vs Regex Mode

| Scenario | Recommended Mode |
|----------|-----------------|
| Quick full-dataset scan | `--mode regex` (fast, free) |
| Deep extraction with relationship fields | `--mode llm` |
| Testing LLM accuracy before full run | `--mode llm --sample 10` |
| CI/CD pipeline QA gate | `--mode regex` |
| Investigating specific charges | `--mode llm --charge DAC-000001-2023` |

---

### Scaling Up

```bash
# Test with 10
python3 qa_charge_report_extraction.py --mode llm --sample 10

# Expand to 100
python3 qa_charge_report_extraction.py --mode llm --sample 100

# Full run with CSV output
python3 qa_charge_report_extraction.py --mode llm --output-csv llm_extracted.csv --issues-csv llm_issues.csv
```

Note: Running all 5,411 reports through the LLM will make 5,411 API calls. Use `--sample` to validate quality before committing to a full run.


In [0]:
import sys
# Manually set the arguments. 
# The first element is the script name, followed by your required flags.
sys.argv = ['qa_charge_report_extraction.py', '--mode', 'llm', '--sample', '10' ] 

try:    exec(open('/Workspace/Users/shawn_wang@tech.gov.sg/pipeline/QA/qa_charge_report_extraction.py').read())
except SystemExit as e:
    pass



Using LLM mode with model: agc-llmaas-claude-4-5-dev
Endpoint: https://litellm.ads-poc.agc.gov.sg/
  CHARGE REPORT EXTRACTION QA REPORT
Total report files found: 5411
Reports processed:        10
Extraction failures:      0
Total CSV rows generated: 18
Reports with issues:      0
  Errors:                 0
  Warnings:               0

All reports passed validation. No issues found.



##Running QA on Charge Reports using RegEx

## QA Charge Report Extraction — Process Explanation

### Purpose

The command `python3 qa_charge_report_extraction.py` performs **automated Quality Assurance (QA) validation** on charge report documents as part of the **medallion framework** (Bronze → Silver layer) data integration pipeline. It scans all `.md` charge report files, extracts structured data using regex patterns, validates data completeness/quality, and produces outputs for downstream processing.

---

### Process Flow

```
ChargeReports/*.md  →  [Regex Extraction]  →  [Validation]  →  QA Report + CSV
    (Bronze)              (Bronze → Silver)      (QA Gate)       (Silver output)
```

### Step-by-Step Breakdown

#### 1. Discovery
The script scans the `ChargeReports/` directory and collects all `.md` files (5,411 charge reports across 2023–2025).

#### 2. Extraction (per file)
For each charge report, regex patterns parse the markdown structure and extract these fields:

| Field | Source in Document | Example |
|-------|-------------------|---------|
| **Case** | `Charge Details (SC-XXXXXX-YYYY / ...)` | `SC-000001-2023` |
| **Charge** | Filename | `DAC-000001-2023` |
| **Entity (Accused)** | Accused Particulars → Name | `Kok Leong Bin Jamal` |
| **Age** | Accused Particulars → Gender/Age | `40` |
| **Gender** | Accused Particulars → Gender/Age | `M` |
| **Entity (Victim)** | Victim Particulars → Name (if present) | `Xiao Ling Teo` |
| **Offence Groups** | Derived from Statute + Offence Section via rule-based mapping | `Traffic Offences` |
| **Special Type** | Special Type field (if present) | `Family Violence` |

The **Offence Groups** classification uses two mapping layers:
- **Statute keywords** (e.g., "Road Traffic Act" → "Traffic Offences")
- **Penal Code section numbers** (e.g., Section 379 → "Theft", Section 323 → "Hurt")

#### 3. Validation
Each extracted record is validated for:

| Check | Severity | Rule |
|-------|----------|------|
| Required fields present | ERROR | Case, Charge, Name, Age, Gender, Offence Group must not be null |
| Case number format | WARNING | Must match `SC-XXXXXX-YYYY` pattern |
| Charge number format | WARNING | Must match `DAC/MAC/MCN-XXXXXX-YYYY` |
| Age range | WARNING | Must be between 7–120 |
| Gender value | ERROR | Must be "M" or "F" |
| Victim field consistency | ERROR | If victim name exists, age and gender must also exist |

#### 4. Output
The script produces:
- **QA Report** (stdout) — summary of total reports processed, issues found, and detailed listing of every error/warning
- **Extracted CSV** (`--output-csv`) — structured data in the same schema as "Info to Extract", with one row per accused and one per victim
- **Issues CSV** (`--issues-csv`) — all validation failures for analysis/dashboarding

---

### Extraction Modes

| Mode | Command | Use Case |
|------|---------|----------|
| **Regex** (default) | `python3 qa_charge_report_extraction.py` | Fast, deterministic, no API cost |
| **LLM** | `python3 qa_charge_report_extraction.py --mode llm` | Richer extraction (e.g., accused's relationship to victim), uses LiteLLM API |

---

### Medallion Framework Integration

| Layer | Role |
|-------|------|
| **Bronze** | Raw `.md` charge report files stored in Unity Catalog Volumes |
| **Silver** | Extracted structured table produced by this script's extraction logic |
| **QA Gate** | Validation step — records with errors are flagged/quarantined for review |
| **Gold** | Clean, aggregated data for analytics dashboards |

The script acts as the **QA gate between Bronze and Silver**, ensuring that the data extraction process produces complete and correctly formatted records before they enter the curated layer.







In [0]:
import sys
# Manually set the arguments. 
# The first element is the script name, followed by your required flags.
sys.argv = ['qa_charge_report_extraction.py'] 

try:    exec(open('/Workspace/Users/shawn_wang@tech.gov.sg/pipeline/QA/qa_charge_report_extraction.py').read())
except SystemExit as e:
    pass

  Processed 500/5411 reports...
  Processed 1000/5411 reports...
  Processed 1500/5411 reports...
  Processed 2000/5411 reports...
  Processed 2500/5411 reports...
  Processed 3000/5411 reports...
  Processed 3500/5411 reports...
  Processed 4000/5411 reports...
  Processed 4500/5411 reports...
  Processed 5000/5411 reports...


  CHARGE REPORT EXTRACTION QA REPORT
Total report files found: 5411
Reports processed:        5411
Extraction failures:      0
Total CSV rows generated: 9141
Reports with issues:      98
  Errors:                 98
  Warnings:               0

--- Issue Summary by Field ---
  Field                       Errors  Warnings
  -------------------------  -------  --------
  case                            98         0

--- Detailed Issues ---
  [ERROR] DAC-001170-2024 — Missing required field: case
  [ERROR] DAC-001171-2024 — Missing required field: case
  [ERROR] DAC-001172-2024 — Missing required field: case
  [ERROR] DAC-001173-2024 — Missing required field: case
  [ERROR] DAC-001174-2024 — Missing required field: case
  [ERROR] DAC-001175-2024 — Missing required field: case
  [ERROR] DAC-001176-2024 — Missing required field: case
  [ERROR] DAC-001177-2024 — Missing required field: case
  [ERROR] DAC-001178-2024 — Missing required field: case
  [ERROR] DAC-001179-2024 — Missing required 